In [70]:
import pandas as pd
import numpy as np

master = pd.read_csv("master_dataset.csv")

In [71]:
master["utilization_rate"] = (
    master["occupancy"] /
    master["count"]
)

In [72]:
master["revenue"] = (
    master["price"] *
    master["volume"]
)

In [73]:
master["weekday"] = pd.to_datetime(
    master["datetime"]
).dt.dayofweek

In [74]:
master["weekend"] = (
    master["weekday"] >= 5
).astype(int)

In [75]:
master["peak_hour"] = (
    master["hour"]
    .isin([7,8,9,18,19,20])
).astype(int)

In [76]:
def congestion(x):

    if x > 0.8:
        return "High"

    elif x > 0.5:
        return "Medium"

    return "Low"

master["congestion"] = (
    master["utilization_rate"]
    .apply(congestion)
)

In [77]:
master["queue_proxy"] = (
    master["occupancy"] *
    master["duration"]
)

In [78]:
master["fast_ratio"] = (
    master["fast_count"] /
    master["count"]
)

In [79]:

master["revenue_per_session"] = (
    master["revenue"] /
    (master["occupancy"] + 1)
)



master["energy_cost_per_kwh"] = 7


master["profit_per_kwh"] = (
    master["price"]
    -
    master["energy_cost_per_kwh"]
)


master["occupancy_density"] = (
    master["occupancy"]
    /
    master["area"]
)


master["charger_mix"] = (
    master["fast_count"]
    /
    (master["slow_count"] + 1)
)


master["peak_congestion_score"] = (
    master["utilization_rate"]
    *
    master["queue_proxy"]
)

In [80]:
master["revenue_per_session"] = (
    master["revenue"] /
    (master["occupancy"] + 1)
)

# Assumption:
# Electricity procurement cost = ₹7 per kWh

master["energy_cost_per_kwh"] = 7


master["profit_per_kwh"] = (
    master["price"]
    -
    master["energy_cost_per_kwh"]
)


master["occupancy_density"] = (
    master["occupancy"]
    /
    master["area"]
)


master["charger_mix"] = (
    master["fast_count"]
    /
    (master["slow_count"] + 1)
)


master["peak_congestion_score"] = (
    master["utilization_rate"]
    *
    master["queue_proxy"]
)

In [81]:
new_features = [
    "revenue_per_session",
    "energy_cost_per_kwh",
    "profit_per_kwh",
    "occupancy_density",
    "charger_mix",
    "peak_congestion_score"
]

print(master[new_features].head())

   revenue_per_session  energy_cost_per_kwh  profit_per_kwh  \
0             0.203162                    7          -6.076   
1             0.310962                    7          -6.076   
2             0.310962                    7          -6.076   
3             0.310962                    7          -6.076   
4             0.310962                    7          -6.076   

   occupancy_density  charger_mix  peak_congestion_score  
0          16.901408     0.107143                  2.352  
1          16.901408     0.107143                  3.600  
2          16.901408     0.107143                  3.600  
3          16.901408     0.107143                  3.600  
4          16.901408     0.107143                  3.600  


In [82]:
print(master.shape)

print(master.columns.tolist())

(2134080, 36)
['timestamp', 'station_id', 'price', 'occupancy', 'volume', 'duration', 'month', 'day', 'year', 'hour', 'minute', 'second', 'datetime', 'num', 'count', 'fast_count', 'slow_count', 'area', 'lon', 'la', 'CBD', 'dynamic_pricing', 'utilization_rate', 'revenue', 'weekday', 'weekend', 'peak_hour', 'congestion', 'queue_proxy', 'fast_ratio', 'revenue_per_session', 'energy_cost_per_kwh', 'profit_per_kwh', 'occupancy_density', 'charger_mix', 'peak_congestion_score']


In [83]:
def period_type(hour):

    if hour in [7,8,9,18,19,20]:
        return "Peak"

    elif hour in [10,11,12,13,14,15,16,17]:
        return "Shoulder"

    else:
        return "Off-Peak"

master["period_type"] = (
    master["hour"]
    .apply(period_type)
)

In [84]:
master["user_type_proxy"] = np.where(
    (
        (master["utilization_rate"] > master["utilization_rate"].median())
        &
        (master["duration"] > master["duration"].median())
    ),
    "Fleet",
    "Public"
)

In [85]:
master["user_type_proxy"].value_counts()

user_type_proxy
Public    1393970
Fleet      740110
Name: count, dtype: int64

In [87]:
master.to_csv(
    "featured_dataset_v2.csv",
    index=False
)

print("Saved Successfully")
print(master.shape)

Saved Successfully
(2134080, 38)
